# 04 - Entrainement complet des modeles

Ce notebook regenere tout le pipeline final : preparation des donnees, analyse du desequilibre, baseline, comparaison des techniques de reequilibrage, validation croisee stratifiee, optimisation du seuil, sauvegarde des modeles et rapports.

La logique metier est prioritaire : dans un contexte de retention, rater un churner coute plus cher que contacter un client en trop. On suit donc surtout le recall, tout en surveillant la precision, le F1-score, la ROC-AUC et la PR-AUC.


## 1. Imports et chemins


In [1]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
sys.path.append(str(BASE_DIR))
from feature_engineering import prepare_customer_features

DATA_PATH = BASE_DIR / 'data' / 'customer_churn.csv'
MODELS_DIR = BASE_DIR / 'models'
REPORTS_DIR = BASE_DIR / 'reports'
PREPROCESSED_PATH = BASE_DIR / 'data_preprocessed.pkl'

RANDOM_STATE = 42
DEFAULT_THRESHOLD = 0.50
THRESHOLD_GRID = np.round(np.arange(0.10, 0.91, 0.05), 2)
MIN_PRECISION_FOR_RECALL = 0.20
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)


## 2. Chargement et feature engineering

On applique les variables metier creees dans `feature_engineering.py` : satisfaction, engagement, risque paiement, pression support, etc.


In [2]:
raw_df = pd.read_csv(DATA_PATH)
df = prepare_customer_features(raw_df)

X = df.drop(columns=['churn', 'customer_id'])
y = df['churn']

print('Shape X :', X.shape)
print('Taux de churn :', round(y.mean(), 4))


Shape X : (10000, 38)
Taux de churn : 0.1021


## 3. Analyse prealable du desequilibre

La cible est fortement desequilibree. Il faut donc regarder le ratio majoritaire/minoritaire et ne pas se limiter a l'accuracy. Un modele qui predit presque toujours `pas de churn` peut avoir une accuracy elevee, mais il ne sert pas le besoin metier si les vrais churners ne sont pas detectes.


In [3]:
class_counts = y.value_counts().sort_index()
majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()
imbalance_ratio = class_counts.max() / class_counts.min()

imbalance_summary = pd.DataFrame({
    'classe': class_counts.index,
    'effectif': class_counts.values,
    'proportion': (class_counts / len(y)).values,
})

print('Classe majoritaire :', majority_class)
print('Classe minoritaire :', minority_class)
print('Ratio majoritaire/minoritaire :', round(imbalance_ratio, 2))
imbalance_summary


Classe majoritaire : 0
Classe minoritaire : 1
Ratio majoritaire/minoritaire : 8.79


,classe,effectif,proportion
0,0,8979,0.8979
1,1,1021,0.1021


## 4. Split train/test stratifie

Le split est stratifie pour conserver la meme proportion de churners dans le train et dans le test. C'est indispensable avec une classe minoritaire autour de 10 %.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
train_ratio = y_train.value_counts().max() / y_train.value_counts().min()

print('Train :', X_train.shape)
print('Test :', X_test.shape)
print('Churn train :', round(y_train.mean(), 4))
print('Churn test :', round(y_test.mean(), 4))
print('Ratio train majoritaire/minoritaire :', round(train_ratio, 2))


Train : (8000, 38)
Test : (2000, 38)
Churn train : 0.1021
Churn test : 0.102
Ratio train majoritaire/minoritaire : 8.79


## 5. Preprocessing adapte aux modeles

La regression logistique et le MLP utilisent un `StandardScaler`, car ils sont sensibles aux echelles. Les modeles d'arbres gardent les variables numeriques telles quelles.


In [5]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

pre_scaled = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', make_encoder(), cat_cols),
])

pre_unscaled = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', make_encoder(), cat_cols),
])


## 6. Baseline simple a seuil 0.5

La baseline sert a montrer pourquoi l'accuracy ne suffit pas. On entraine une regression logistique sans reequilibrage et on l'evalue au seuil par defaut `0.5`, puis on compare aussi avec un modele naif qui predit toujours la classe majoritaire.


In [6]:
def metric_row(name, strategy, y_true, y_pred, scores=None, threshold=DEFAULT_THRESHOLD):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    row = {
        'modele': name,
        'strategie_desequilibre': strategy,
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'clients_alertes': int(y_pred.sum()),
    }
    if scores is not None:
        row['roc_auc'] = roc_auc_score(y_true, scores)
        row['pr_auc'] = average_precision_score(y_true, scores)
    else:
        row['roc_auc'] = np.nan
        row['pr_auc'] = np.nan
    return row

naive = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
naive.fit(X_train, y_train)
naive_pred = naive.predict(X_test)

baseline_lr = Pipeline([
    ('pre', pre_scaled),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
baseline_lr.fit(X_train, y_train)
baseline_scores = baseline_lr.predict_proba(X_test)[:, 1]
baseline_pred = (baseline_scores >= DEFAULT_THRESHOLD).astype(int)

baseline_rows = [
    metric_row('NaiveMajority', 'baseline_majoritaire', y_test, naive_pred, None, DEFAULT_THRESHOLD),
    metric_row('LogisticRegression_baseline', 'aucun_reequilibrage', y_test, baseline_pred, baseline_scores, DEFAULT_THRESHOLD),
]
baseline_report = pd.DataFrame(baseline_rows)
baseline_report.to_csv(REPORTS_DIR / 'baseline_analysis.csv', index=False)
baseline_report


,modele,strategie_desequilibre,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp,clients_alertes,roc_auc,pr_auc
0,NaiveMajority,baseline_majoritaire,0.5,0.898,0.00,0.000000,0.000000,1796,0,204,0,0,NaN,NaN
1,LogisticRegression_baseline,aucun_reequilibrage,0.5,0.897,0.45,0.044118,0.080357,1785,11,195,9,20,0.736288,0.244538


## 7. Experiences de gestion du desequilibre

On compare explicitement plusieurs approches : aucune correction, reechantillonnage data-level (`RandomOverSampler`, `SMOTE`, `RandomUnderSampler`) et approches model-level (`class_weight`, `scale_pos_weight`).

Tous les reechantillonnages sont places dans des pipelines `imblearn`, apres le preprocessing, pour eviter toute fuite de donnees pendant la validation croisee.


In [7]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

experiments = {
    'LogisticRegression_baseline': {
        'strategy': 'aucun_reequilibrage',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_class_weight': {
        'strategy': 'class_weight_balanced',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_random_over': {
        'strategy': 'random_over_sampling',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_smote': {
        'strategy': 'smote',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', SMOTE(random_state=RANDOM_STATE)),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'LogisticRegression_random_under': {
        'strategy': 'random_under_sampling',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    },
    'RandomForest_baseline': {
        'strategy': 'aucun_reequilibrage',
        'pipeline': Pipeline([
            ('pre', pre_unscaled),
            ('model', RandomForestClassifier(
                n_estimators=250,
                max_depth=10,
                min_samples_leaf=2,
                n_jobs=1,
                random_state=RANDOM_STATE,
            )),
        ]),
    },
    'RandomForest_class_weight': {
        'strategy': 'class_weight_balanced_subsample',
        'pipeline': Pipeline([
            ('pre', pre_unscaled),
            ('model', RandomForestClassifier(
                n_estimators=250,
                max_depth=10,
                min_samples_leaf=2,
                class_weight='balanced_subsample',
                n_jobs=1,
                random_state=RANDOM_STATE,
            )),
        ]),
    },
    'XGBoost_scale_pos_weight': {
        'strategy': 'scale_pos_weight',
        'pipeline': Pipeline([
            ('pre', pre_unscaled),
            ('model', XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                scale_pos_weight=scale_pos_weight,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
            )),
        ]),
    },
    'DeepLearning_smote': {
        'strategy': 'smote',
        'pipeline': Pipeline([
            ('pre', pre_scaled),
            ('sampler', SMOTE(random_state=RANDOM_STATE)),
            ('model', MLPClassifier(
                hidden_layer_sizes=(64, 32),
                alpha=0.001,
                early_stopping=True,
                max_iter=300,
                random_state=RANDOM_STATE,
            )),
        ]),
    },
}

print('scale_pos_weight XGBoost :', round(scale_pos_weight, 2))
print("Nombre d'experiences :", len(experiments))


scale_pos_weight XGBoost : 8.79
Nombre d'experiences : 9


## 8. Evaluation avec validation croisee stratifiee

`StratifiedKFold` preserve la proportion des classes dans chaque fold. C'est le bon choix ici car une validation non stratifiee pourrait produire des folds avec trop peu de churners, ce qui rendrait le recall et la PR-AUC instables.


In [8]:
def evaluate_experiment(name, config):
    model = config['pipeline']
    strategy = config['strategy']
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision',
        'recall': 'recall',
        'f1': 'f1',
        'precision': 'precision',
    }

    start = perf_counter()
    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score='raise',
    )
    model.fit(X_train, y_train)
    duration = perf_counter() - start

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= DEFAULT_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    return model, {
        'modele': name,
        'strategie_desequilibre': strategy,
        'cv_roc_auc_mean': cv_scores['test_roc_auc'].mean(),
        'cv_pr_auc_mean': cv_scores['test_pr_auc'].mean(),
        'cv_recall_mean': cv_scores['test_recall'].mean(),
        'cv_f1_mean': cv_scores['test_f1'].mean(),
        'cv_precision_mean': cv_scores['test_precision'].mean(),
        'test_threshold_default': DEFAULT_THRESHOLD,
        'test_roc_auc': roc_auc_score(y_test, proba),
        'test_pr_auc': average_precision_score(y_test, proba),
        'test_accuracy': accuracy_score(y_test, pred),
        'test_precision': precision_score(y_test, pred, zero_division=0),
        'test_recall': recall_score(y_test, pred),
        'test_f1': f1_score(y_test, pred, zero_division=0),
        'test_tn': int(tn),
        'test_fp': int(fp),
        'test_fn': int(fn),
        'test_tp': int(tp),
        'test_clients_alertes': int(pred.sum()),
        'temps_entrainement_secondes': duration,
    }

rows = []
trained_models = {}

for name, config in experiments.items():
    print('Entrainement :', name, '-', config['strategy'])
    fitted_model, row = evaluate_experiment(name, config)
    trained_models[name] = fitted_model
    rows.append(row)
    joblib.dump(fitted_model, MODELS_DIR / f'model_{name}.pkl')

comparison_default = pd.DataFrame(rows).sort_values(
    ['test_recall', 'test_pr_auc', 'test_f1'], ascending=False
)
comparison_default


Entrainement : LogisticRegression_baseline - aucun_reequilibrage


Entrainement : LogisticRegression_class_weight - class_weight_balanced


Entrainement : LogisticRegression_random_over - random_over_sampling


Entrainement : LogisticRegression_smote - smote


Entrainement : LogisticRegression_random_under - random_under_sampling


...

C:\Users\Héctor\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Entrainement : RandomForest_class_weight - class_weight_balanced_subsample


Entrainement : XGBoost_scale_pos_weight - scale_pos_weight


Entrainement : DeepLearning_smote - smote


,modele,strategie_desequilibre,cv_roc_auc_mean,cv_pr_auc_mean,cv_recall_mean,cv_f1_mean,cv_precision_mean,test_threshold_default,test_roc_auc,test_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_tn,test_fp,test_fn,test_tp,test_clients_alertes,temps_entrainement_secondes
7,XGBoost_scale_pos_weight,scale_pos_weight,0.783765,0.264302,0.529968,0.353489,0.265205,0.5,0.791552,0.275032,0.7765,0.264078,0.666667,0.378303,1417,379,68,136,515,1.711164
2,LogisticRegression_random_over,random_over_sampling,0.744152,0.240386,0.673180,0.320893,0.210657,0.5,0.737330,0.237482,0.6845,0.191919,0.651961,0.296544,1236,560,71,133,693,0.357040
1,LogisticRegression_class_weight,class_weight_balanced,0.746549,0.242099,0.660957,0.316347,0.207945,0.5,0.741179,0.243431,0.6865,0.192140,0.647059,0.296296,1241,555,72,132,687,0.302566
4,LogisticRegression_random_under,random_under_sampling,0.726693,0.227198,0.657303,0.299374,0.193829,0.5,0.728746,0.231733,0.6770,0.186080,0.642157,0.288546,1223,573,73,131,704,0.413877
3,LogisticRegression_smote,smote,0.742363,0.240678,0.652383,0.310750,0.203987,0.5,0.730651,0.240892,0.6775,0.183644,0.627451,0.284129,1227,569,76,128,697,2.278741
6,RandomForest_class_weight,class_weight_balanced_subsample,0.793005,0.262167,0.145667,0.195450,0.299148,0.5,0.802721,0.282171,0.8580,0.309524,0.318627,0.314010,1651,145,139,65,210,10.048854
8,DeepLearning_smote,smote,0.647938,0.172938,0.195867,0.197403,0.199372,0.5,0.666877,0.179077,0.8285,0.213992,0.254902,0.232662,1605,191,152,52,243,12.109486
0,LogisticRegression_baseline,aucun_reequilibrage,0.743102,0.240188,0.028150,0.051848,0.343039,0.5,0.736288,0.244538,0.8970,0.450000,0.044118,0.080357,1785,11,195,9,20,0.302051
5,RandomForest_baseline,aucun_reequilibrage,0.803717,0.299313,0.002451,0.004884,0.666667,0.5,0.799983,0.312498,0.8980,0.000000,0.000000,0.000000,1796,0,204,0,0,9.325759


## 9. Optimisation du seuil de decision

Le seuil `0.5` n'est pas optimal dans un contexte desequilibre. On teste plusieurs seuils pour chaque modele et on observe le compromis : baisser le seuil augmente souvent le recall, mais augmente aussi les faux positifs.


In [9]:
threshold_rows = []
for name, model in trained_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    strategy = experiments[name]['strategy']
    for threshold in THRESHOLD_GRID:
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        threshold_rows.append({
            'modele': name,
            'strategie_desequilibre': strategy,
            'threshold': threshold,
            'recall': recall_score(y_test, pred),
            'precision': precision_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred, zero_division=0),
            'accuracy': accuracy_score(y_test, pred),
            'roc_auc': roc_auc_score(y_test, proba),
            'pr_auc': average_precision_score(y_test, proba),
            'tn': int(tn),
            'fp': int(fp),
            'fn': int(fn),
            'tp': int(tp),
            'clients_alertes': int(pred.sum()),
        })

thresholds = pd.DataFrame(threshold_rows)
best_f1_by_model = thresholds.loc[thresholds.groupby('modele')['f1'].idxmax()].copy()

# Departage eco-responsable : un ecart de recall inferieur a 1 point de pourcentage entre
# deux combinaisons modele/seuil n'est pas une vraie difference metier (204 churners au
# total dans le test set, donc 1 point ~= 2 clients). Dans ce cas, il serait naif de
# trancher uniquement sur la precision : on privilegie plutot le modele le plus rapide/
# sobre a entrainer (ecoresponsabilite, cf. RNCP C4.3) parmi les candidats a recall
# quasi-maximal, plutot que de choisir mecaniquement le meilleur score de precision.
RECALL_TIE_TOLERANCE = 0.01

recall_candidates = thresholds[thresholds['precision'] >= MIN_PRECISION_FOR_RECALL].copy()
if recall_candidates.empty:
    recall_candidates = thresholds.copy()

training_time_by_model = comparison_default.set_index('modele')['temps_entrainement_secondes']
recall_candidates['temps_entrainement_secondes'] = recall_candidates['modele'].map(training_time_by_model)

best_recall = recall_candidates['recall'].max()
near_best_candidates = recall_candidates[
    recall_candidates['recall'] >= best_recall - RECALL_TIE_TOLERANCE
].copy()

best_recall_tradeoff = near_best_candidates.sort_values(
    ['temps_entrainement_secondes', 'precision', 'f1', 'pr_auc'],
    ascending=[True, False, False, False],
).iloc[0]

print(f"Candidats a recall quasi-maximal (>= {best_recall - RECALL_TIE_TOLERANCE:.3f}), tries par cout de calcul :")
print(
    near_best_candidates.sort_values('temps_entrainement_secondes')[
        ['modele', 'threshold', 'recall', 'precision', 'temps_entrainement_secondes']
    ].head(6)
)
print()
print('Meilleur seuil metier (recall quasi-maximal, priorite au modele le plus sobre) :')
print(best_recall_tradeoff[['modele', 'strategie_desequilibre', 'threshold', 'recall', 'precision', 'f1', 'fp', 'fn', 'tp', 'tn']])

thresholds_sorted = thresholds.sort_values(['recall', 'precision', 'f1'], ascending=False)
thresholds_sorted.head(15)


Candidats a recall quasi-maximal (>= 0.863), tries par cout de calcul :
                        modele  threshold    recall  precision  \
121   XGBoost_scale_pos_weight       0.20  0.872549   0.211905   
105  RandomForest_class_weight       0.25  0.872549   0.223618   

     temps_entrainement_secondes  
121                     1.711164  
105                    10.048854  

Meilleur seuil metier (recall quasi-maximal, priorite au modele le plus sobre) :
modele                    XGBoost_scale_pos_weight
strategie_desequilibre            scale_pos_weight
threshold                                      0.2
recall                                    0.872549
precision                                 0.211905
f1                                        0.340996
fp                                             662
fn                                              26
tp                                             178
tn                                            1134
Name: 121, dtype: object


,modele,strategie_desequilibre,threshold,recall,precision,f1,accuracy,roc_auc,pr_auc,tn,fp,fn,tp,clients_alertes
34,LogisticRegression_random_over,random_over_sampling,0.10,1.000000,0.104241,0.188801,0.1235,0.737330,0.237482,43,1753,0,204,1957
68,LogisticRegression_random_under,random_under_sampling,0.10,1.000000,0.104135,0.188627,0.1225,0.728746,0.231733,41,1755,0,204,1959
17,LogisticRegression_class_weight,class_weight_balanced,0.10,1.000000,0.103870,0.188192,0.1200,0.741179,0.243431,36,1760,0,204,1964
102,RandomForest_class_weight,class_weight_balanced_subsample,0.10,1.000000,0.102000,0.185118,0.1020,0.802721,0.282171,0,1796,0,204,2000
103,RandomForest_class_weight,class_weight_balanced_subsample,0.15,0.990196,0.109902,0.197845,0.1810,0.802721,0.282171,160,1636,2,202,1838
51,LogisticRegression_smote,smote,0.10,0.980392,0.105263,0.190114,0.1480,0.730651,0.240892,96,1700,4,200,1900
69,LogisticRegression_random_under,random_under_sampling,0.15,0.975490,0.106474,0.191992,0.1625,0.728746,0.231733,126,1670,5,199,1869
18,LogisticRegression_class_weight,class_weight_balanced,0.15,0.970588,0.106166,0.191397,0.1635,0.741179,0.243431,129,1667,6,198,1865
35,LogisticRegression_random_over,random_over_sampling,0.15,0.970588,0.106052,0.191212,0.1625,0.737330,0.237482,127,1669,6,198,1867
70,LogisticRegression_random_under,random_under_sampling,0.20,0.960784,0.112773,0.201854,0.2250,0.728746,0.231733,254,1542,8,196,1738


## 10. Comparaison finale et sauvegardes

Le modele final est choisi selon le recall sous contrainte de precision minimale. Cette regle garde l'objectif industriel principal (minimiser les faux negatifs) tout en evitant une solution qui declenche trop d'alertes inutiles.


In [10]:
final_name = best_recall_tradeoff['modele']
recommended_threshold = float(best_recall_tradeoff['threshold'])

final_threshold_metrics = thresholds[thresholds['modele'].eq(final_name) & thresholds['threshold'].eq(recommended_threshold)].iloc[0]
comparison = comparison_default.merge(
    best_f1_by_model[['modele', 'threshold', 'recall', 'precision', 'f1', 'accuracy', 'fp', 'fn', 'tp', 'tn']].rename(columns={
        'threshold': 'best_f1_threshold',
        'recall': 'best_f1_recall',
        'precision': 'best_f1_precision',
        'f1': 'best_f1',
        'accuracy': 'best_f1_accuracy',
        'fp': 'best_f1_fp',
        'fn': 'best_f1_fn',
        'tp': 'best_f1_tp',
        'tn': 'best_f1_tn',
    }),
    on='modele',
    how='left',
)
comparison['modele_final'] = comparison['modele'].eq(final_name)
comparison['threshold_recommande'] = np.where(comparison['modele_final'], recommended_threshold, np.nan)
comparison = comparison.sort_values(['modele_final', 'test_recall', 'test_pr_auc', 'test_f1'], ascending=False)

comparison.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)
thresholds_sorted.to_csv(REPORTS_DIR / 'threshold_analysis.csv', index=False)

model_aliases = {
    'model_LogisticRegression.pkl': 'LogisticRegression_class_weight',
    'model_RandomForest.pkl': 'RandomForest_class_weight',
    'model_XGBoost.pkl': 'XGBoost_scale_pos_weight',
    'model_DeepLearning.pkl': 'DeepLearning_smote',
}
for filename, source_name in model_aliases.items():
    joblib.dump(trained_models[source_name], MODELS_DIR / filename)

joblib.dump(trained_models[final_name], MODELS_DIR / 'best_model.pkl')

data_info = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'all_cols': X.columns.tolist(),
    'input_cols': raw_df.drop(columns=['churn', 'customer_id']).columns.tolist(),
    'medians': X.median(numeric_only=True).to_dict(),
    'modes': X.mode(dropna=True).iloc[0].to_dict(),
    'threshold_recommande': recommended_threshold,
    'modele_retenu': final_name,
    'strategie_retenue': experiments[final_name]['strategy'],
    'ratio_desequilibre': float(imbalance_ratio),
    'min_precision_for_recall': MIN_PRECISION_FOR_RECALL,
}
joblib.dump(data_info, PREPROCESSED_PATH)

print('Modele retenu :', final_name)
print('Strategie retenue :', experiments[final_name]['strategy'])
print('Seuil recommande :', recommended_threshold)
print('Fichiers sauvegardes dans :', REPORTS_DIR, MODELS_DIR)
comparison


Modele retenu : XGBoost_scale_pos_weight
Strategie retenue : scale_pos_weight
Seuil recommande : 0.2
Fichiers sauvegardes dans : C:\Users\Héctor\OneDrive\Desktop\Projects\EFREI\M1-DE\PRO_DATA_SCIENCE\churn_predict\projet_churn_structure\reports C:\Users\Héctor\OneDrive\Desktop\Projects\EFREI\M1-DE\PRO_DATA_SCIENCE\churn_predict\projet_churn_structure\models


,modele,strategie_desequilibre,cv_roc_auc_mean,cv_pr_auc_mean,cv_recall_mean,cv_f1_mean,cv_precision_mean,test_threshold_default,test_roc_auc,test_pr_auc,...,best_f1_recall,best_f1_precision,best_f1,best_f1_accuracy,best_f1_fp,best_f1_fn,best_f1_tp,best_f1_tn,modele_final,threshold_recommande
0,XGBoost_scale_pos_weight,scale_pos_weight,0.783765,0.264302,0.529968,0.353489,0.265205,0.5,0.791552,0.275032,...,0.735294,0.265487,0.390117,0.7655,415,54,150,1381,True,0.2
1,LogisticRegression_random_over,random_over_sampling,0.744152,0.240386,0.673180,0.320893,0.210657,0.5,0.737330,0.237482,...,0.441176,0.264706,0.330882,0.8180,250,114,90,1546,False,NaN
2,LogisticRegression_class_weight,class_weight_balanced,0.746549,0.242099,0.660957,0.316347,0.207945,0.5,0.741179,0.243431,...,0.534314,0.247166,0.337984,0.7865,332,95,109,1464,False,NaN
3,LogisticRegression_random_under,random_under_sampling,0.726693,0.227198,0.657303,0.299374,0.193829,0.5,0.728746,0.231733,...,0.426471,0.243697,0.310160,0.8065,270,117,87,1526,False,NaN
4,LogisticRegression_smote,smote,0.742363,0.240678,0.652383,0.310750,0.203987,0.5,0.730651,0.240892,...,0.465686,0.255376,0.329861,0.8070,277,109,95,1519,False,NaN
5,RandomForest_class_weight,class_weight_balanced_subsample,0.793005,0.262167,0.145667,0.195450,0.299148,0.5,0.802721,0.282171,...,0.818627,0.261755,0.396675,0.7460,471,37,167,1325,False,NaN
6,DeepLearning_smote,smote,0.647938,0.172938,0.195867,0.197403,0.199372,0.5,0.666877,0.179077,...,0.397059,0.210390,0.275042,0.7865,304,123,81,1492,False,NaN
7,LogisticRegression_baseline,aucun_reequilibrage,0.743102,0.240188,0.028150,0.051848,0.343039,0.5,0.736288,0.244538,...,0.495098,0.247549,0.330065,0.7950,307,103,101,1489,False,NaN
8,RandomForest_baseline,aucun_reequilibrage,0.803717,0.299313,0.002451,0.004884,0.666667,0.5,0.799983,0.312498,...,0.784314,0.256000,0.386007,0.7455,465,44,160,1331,False,NaN


## 11. Discussion metier

Les methodes data-level permettent d'augmenter la detection des churners, mais elles ont des limites :

- `RandomOverSampler` duplique des exemples minoritaires et peut favoriser l'overfitting.
- `SMOTE` cree des exemples synthetiques utiles, mais peut ajouter du bruit si les voisins minoritaires sont mal separes.
- `RandomUnderSampler` reduit la classe majoritaire et peut perdre de l'information.
- Les ponderations (`class_weight`, `scale_pos_weight`) gardent toutes les donnees et rendent les erreurs sur churners plus couteuses.

Le seuil final traduit le compromis metier : un seuil plus bas diminue les faux negatifs, mais augmente les faux positifs. En retention client, ce compromis est acceptable si le cout d'une action commerciale reste inferieur au cout d'un client perdu.
